In [1]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
# Permite importar el paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

# 06 · Optimización con seguimiento (tracking)

Este notebook **solo orquesta**: los espacios de búsqueda, la búsqueda con registro, el intervalo por anfitrión y las reglas
de decisión están en `src/tarifalatam/models/tuning.py`, con sus pruebas en `tests/test_tuning.py`. La conexión con MLflow
es el módulo que ya existía, `src/tarifalatam/tracking/mlflow_utils.py`, **sin reescribirlo**.

**Reglas escritas antes de buscar** (`especificaciones/04-features-torneo-optimizacion.md`, sección 6, aprobada el
2026-10-31):

1. Se optimizan **solo los finalistas** del torneo (`config.FINALISTS`), con las variables de la etapa 4 (`config.FINAL_IDEAS`).
2. **Presupuesto: 30 ensayos por finalista**, fijado antes de empezar. Búsqueda bayesiana (Optuna, `TPESampler` con semilla
   42) que minimiza el MAE medio en USD.
3. **El 25 % de los anfitriones se reserva** (`GroupShuffleSplit`, semilla 7) y la búsqueda nunca lo ve. La búsqueda usa los
   pliegues de `folds.csv` restringidos a sus anfitriones.
4. **Un ensayo = una ejecución** en el MLflow **local** (`configure_mlflow(force_local=True)`). Databricks entra en la clase 4.
5. **La mejor puntuación de la búsqueda es optimista** (el máximo de muchas estimaciones con ruido): no se reporta como
   resultado. La decisión se toma en los anfitriones reservados, con un intervalo del 95 % que remuestrea **anfitriones
   completos**. **Se usan los parámetros optimizados solo si todo el intervalo está por debajo de cero.**
6. **Entre dos finalistas** pasa el que se equivoca menos en todo el intervalo; si cruza el cero, el más rápido de ajustar
   con su configuración elegida (`choose_finalist`).

**La prueba no se abre.** **Salidas:** `reports/06_optimizacion_tabla.csv` (un ensayo por fila) y
`reports/06_best_params.json` (la decisión).

## 0. Seguimiento local

In [2]:
from tarifalatam import config
from tarifalatam.config import EXPERIMENT_NAME, GROUP_COL, N_TRIALS, REPORTS_DIR, TARGET
from tarifalatam.tracking.mlflow_utils import configure_mlflow, set_experiment

with warnings.catch_warnings(record=True) as caught:  # se muestra por qué se usó (o no) Databricks
    warnings.simplefilter("always")
    where = configure_mlflow(force_local=True)  # clase 3: seguimiento local, sin Databricks ni tokens
for warning in caught:
    print("Aviso:", warning.message)
experiment = set_experiment(EXPERIMENT_NAME)
print("MLflow:", where, "| experimento:", experiment)

2026/10/31 11:32:50 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/31 11:32:50 INFO mlflow.store.db.utils: Updating database tables


2026/10/31 11:32:51 INFO mlflow.tracking.fluent: Experiment with name 'tarifalatam-optimizacion' does not exist. Creating a new experiment.


MLflow: local | experimento: tarifalatam-optimizacion


## 1. Reservar anfitriones que la búsqueda no verá

In [3]:
from tarifalatam.data.folds import fold_indices, load_folds, reserve_hosts
from tarifalatam.data.load import load_train
from tarifalatam.models.train import feature_frame

train = load_train()
folds = load_folds()
fold_indices(train, folds)  # comprueba que la asignación de pliegues coincide con el entrenamiento
search, reserved = reserve_hosts(train)
assert set(search[GROUP_COL]).isdisjoint(reserved[GROUP_COL]), "hay anfitriones en los dos lados"
search_folds = fold_indices(search, folds, exact=False)  # los mismos pliegues, solo con anfitriones de búsqueda
print(f"búsqueda: {len(search):,} anuncios de {search[GROUP_COL].nunique():,} anfitriones")
print(f"reservados: {len(reserved):,} anuncios de {reserved[GROUP_COL].nunique():,} anfitriones (la búsqueda no los ve)")
print("anuncios de validación por pliegue en la búsqueda:", [len(v) for _, v in search_folds])

búsqueda: 89,490 anuncios de 40,038 anfitriones
reservados: 29,023 anuncios de 13,347 anfitriones (la búsqueda no los ve)
anuncios de validación por pliegue en la búsqueda: [17830, 18400, 17597, 17889, 17774]


## 2. Presupuesto y espacio de búsqueda (escritos antes de buscar)

| Finalista | Hiperparámetros y rangos |
|---|---|
| LightGBM | `learning_rate` 0,01–0,2 (logarítmica) · `n_estimators` 200–1.000 · `num_leaves` 15–255 · `min_child_samples` 10–200 · `subsample` 0,6–1,0 (`subsample_freq=1`) · `colsample_bytree` 0,5–1,0 · `reg_lambda` 0,001–10 (logarítmica) |
| Bosque aleatorio | `n_estimators` fijo en 200 · `max_depth` 8–40 · `min_samples_leaf` 1–20 · `max_features` 0,2–1,0 |
| Ridge | `alpha` 0,001–1.000 (logarítmica) |

**Presupuesto: 30 ensayos por finalista.**

In [4]:
finalists = config.FINALISTS  # elegidos en la etapa 5 (reports/05_torneo.md)
ideas = config.FINAL_IDEAS  # decididas en la etapa 4 (reports/04_features.md)
print("finalistas:", finalists, "· ensayos por finalista:", N_TRIALS, "· ideas:", ideas or "ninguna (referencia)")

finalistas: ('random_forest', 'lightgbm') · ensayos por finalista: 30 · ideas: ninguna (referencia)


## 3. La búsqueda: un ensayo = una ejecución registrada

Antes de buscar se mide la configuración **de partida** (la del torneo) en los mismos pliegues de búsqueda, como punto de
comparación. Ojo: la mejor puntuación de la búsqueda frente a la de partida es **optimista**, porque es la mejor de 30. La
decisión se toma en la sección 4.

In [5]:
import mlflow

from tarifalatam.models.train import build_pipeline, candidates, evaluate_folds
from tarifalatam.models.tuning import trials_table, tune

X_search, y_search = feature_frame(search), search[TARGET].to_numpy()
start_ms = int(time.time() * 1000)  # para contar solo las ejecuciones de esta corrida
studies, trials, start_cv = {}, [], {}
for name in finalists:
    model, family = candidates()[name]
    start_cv[name], _ = evaluate_folds(build_pipeline(model, family, ideas=ideas), X_search, y_search, search_folds)
    start = time.perf_counter()
    studies[name] = tune(name, X_search, y_search, search_folds, n_trials=N_TRIALS, ideas=ideas)
    trials.append(trials_table(studies[name], name))
    n_complete = sum(t.state.name == "COMPLETE" for t in studies[name].trials)
    print(f"{name}: {len(studies[name].trials)} ensayos ({n_complete} completos) en {(time.perf_counter() - start) / 60:.1f} min · "
          f"MAE de partida {start_cv[name]['mae'].mean():.2f} · mejor ensayo {studies[name].best_value:.2f} USD (optimista)")

trials = pd.concat(trials, ignore_index=True)
trials.to_csv(REPORTS_DIR / "06_optimizacion_tabla.csv", index=False)
this_run = mlflow.search_runs(experiment_names=[experiment], filter_string=f"attributes.start_time >= {start_ms}")
print(f"ejecuciones registradas en esta corrida: {len(this_run)} · ensayos: {len(trials)}")
assert len(this_run) == len(trials), "cada ensayo debe ser exactamente una ejecución"
assert (trials.groupby("model").size() <= N_TRIALS).all(), "se pasó el presupuesto"

random_forest: 30 ensayos (30 completos) en 6.4 min · MAE de partida 40.17 · mejor ensayo 40.03 USD (optimista)


lightgbm: 30 ensayos (30 completos) en 7.6 min · MAE de partida 40.26 · mejor ensayo 39.69 USD (optimista)
ejecuciones registradas en esta corrida: 60 · ensayos: 60


In [6]:
for name in finalists:
    print(f"{name}: los 5 mejores ensayos (MAE medio en los pliegues de búsqueda)")
    best_rows = trials[(trials["model"] == name) & (trials["state"] == "completo")].nsmallest(5, "mae")
    display(best_rows.drop(columns=["model", "state"]).dropna(axis=1, how="all").round(4))

random_forest: los 5 mejores ensayos (MAE medio en los pliegues de búsqueda)


,trial,max_depth,min_samples_leaf,max_features,mae,mae_std,mape,rmse,bias,r2_log,fit_seconds
25,26,24.0,4.0,0.4442,40.0275,2.4601,32.8997,113.5593,-13.9117,0.6381,2.4129
22,23,19.0,5.0,0.4006,40.0333,2.5228,32.8755,113.5701,-14.2864,0.6375,2.0732
16,17,27.0,5.0,0.4834,40.0725,2.5056,32.9628,113.4461,-13.8013,0.6372,2.4980
26,27,20.0,6.0,0.4209,40.0802,2.5335,32.8355,114.2863,-14.4239,0.6374,2.3103
1,2,27.0,4.0,0.3248,40.0877,2.5254,32.9048,113.9570,-14.2615,0.6377,2.0198


lightgbm: los 5 mejores ensayos (MAE medio en los pliegues de búsqueda)


,trial,mae,mae_std,mape,rmse,bias,r2_log,fit_seconds,n_estimators,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_lambda
59,30,39.6895,2.5047,32.3973,111.4510,-13.3696,0.6471,3.0946,522.0,0.0167,215.0,19.0,0.9281,0.5425,2.8855
54,25,39.6959,2.5391,32.4208,111.5721,-12.9507,0.6468,3.7968,753.0,0.0204,165.0,21.0,0.9692,0.6118,9.8372
39,10,39.6977,2.5809,32.3761,111.2584,-13.2503,0.6469,2.6294,485.0,0.0232,145.0,36.0,0.9209,0.5373,8.8623
42,13,39.7026,2.4926,32.3848,111.3766,-13.2471,0.6470,3.1312,559.0,0.0214,157.0,14.0,0.9663,0.5248,3.0581
53,24,39.7190,2.5701,32.4048,111.1172,-13.0458,0.6468,2.9751,680.0,0.0204,133.0,76.0,0.9526,0.5544,2.7452


## 4. La comprobación honesta

Se entrenan dos modelos con **todos** los datos de búsqueda: parámetros **de partida** y **optimizados**. Se comparan en los
anfitriones reservados con la diferencia de error absoluto por anuncio (optimizado − partida, en USD) y un intervalo del 95 %
con 1.000 remuestreos de anfitriones completos (semilla 42).

**Regla escrita antes de mirar:** se usan los optimizados **solo si todo el intervalo está por debajo de cero**. Si cruza el
cero, la mejora no se distingue del ruido y se usan los de partida.

In [7]:
from tarifalatam.models.tuning import FINALIST_FAMILIES, compare_on_reserved, make_model, starting_params, use_optimized

checks, chosen = {}, {}
for name in finalists:
    family = FINALIST_FAMILIES[name]
    start_pipeline = build_pipeline(candidates()[name][0], family, ideas=ideas)  # la configuración del torneo
    tuned_pipeline = build_pipeline(make_model(name, studies[name].best_params), family, ideas=ideas)
    result = compare_on_reserved(tuned_pipeline, start_pipeline, search, reserved)  # A = optimizado, B = partida
    use_tuned = use_optimized(result["low"], result["high"])
    checks[name] = result
    chosen[name] = {
        "origin": "optimizados" if use_tuned else "partida",
        "params": dict(studies[name].best_params) if use_tuned else starting_params(name),
        "pipeline": tuned_pipeline if use_tuned else start_pipeline,
        # segundos de ajuste por pliegue de la configuración elegida, medidos en los pliegues de búsqueda
        "seconds": studies[name].best_trial.user_attrs["fit_seconds"] if use_tuned else start_cv[name]["fit_seconds"].mean(),
    }
    print(f"{name}: MAE en reservados · partida {result['mae_b']:.2f} · optimizado {result['mae_a']:.2f} USD")
    print(f"   diferencia (optimizado − partida) {result['mean_difference']:+.2f} USD · intervalo 95 % "
          f"[{result['low']:+.2f}, {result['high']:+.2f}] · sesgo partida {result['bias_b']:+.2f} · sesgo optimizado {result['bias_a']:+.2f}")
    print(f"   decisión: se usan los parámetros {chosen[name]['origin']} · {chosen[name]['seconds']:.2f} s de ajuste por pliegue")

random_forest: MAE en reservados · partida 38.87 · optimizado 38.70 USD
   diferencia (optimizado − partida) -0.17 USD · intervalo 95 % [-0.28, -0.06] · sesgo partida -13.09 · sesgo optimizado -13.56
   decisión: se usan los parámetros optimizados · 2.41 s de ajuste por pliegue


lightgbm: MAE en reservados · partida 38.86 · optimizado 38.52 USD
   diferencia (optimizado − partida) -0.34 USD · intervalo 95 % [-0.64, -0.01] · sesgo partida -13.45 · sesgo optimizado -12.97
   decisión: se usan los parámetros optimizados · 3.09 s de ajuste por pliegue


### 4.1 Si hay dos finalistas: ¿cuál pasa a la etapa 7?

Se comparan las dos configuraciones elegidas en los mismos anfitriones reservados, con el mismo intervalo. Pasa el que tenga
todo el intervalo de la diferencia por debajo de cero. Si cruza el cero, pasa el más rápido de ajustar **con su configuración
elegida** (`choose_finalist`, segundos por pliegue medidos en la búsqueda).

In [8]:
from tarifalatam.models.tuning import choose_finalist

comparison = None
if len(finalists) == 1:
    rule_winner, rule_reason = finalists[0], "un solo finalista"
else:
    a, b = finalists
    result = compare_on_reserved(chosen[a]["pipeline"], chosen[b]["pipeline"], search, reserved)  # A − B
    seconds = {name: chosen[name]["seconds"] for name in finalists}
    rule_winner, rule_reason = choose_finalist(a, b, result["low"], result["high"], seconds)
    comparison = {"finalista_a": a, "finalista_b": b, "mae_a": result["mae_a"], "mae_b": result["mae_b"],
                  "diferencia_media": result["mean_difference"], "intervalo_95": [result["low"], result["high"]],
                  "segundos_por_pliegue": seconds, "motivo_regla": rule_reason}
    print(f"{a} {result['mae_a']:.2f} frente a {b} {result['mae_b']:.2f} USD · diferencia {result['mean_difference']:+.2f} · "
          f"intervalo [{result['low']:+.2f}, {result['high']:+.2f}]")
print("La regla elige:", rule_winner, "·", rule_reason)

random_forest 38.70 frente a lightgbm 38.52 USD · diferencia +0.17 · intervalo [-0.20, +0.55]
La regla elige: random_forest · el intervalo cruza el cero: empate; pasa el más rápido de ajustar con su configuración elegida (random_forest: 2.41 s por pliegue frente a 3.09 s)


### 4.2 Decisión del grupo

`config.STAGE7_MODEL` guarda el modelo que el grupo decidió pasar a la etapa 7 (tarea T047). Si es distinto de lo que elige
la regla, es una **excepción a la regla escrita, decidida después de ver el resultado**, y queda registrada como tal en el JSON
(`decision_del_grupo`), junto con lo que habría elegido la regla (`regla_elegiria`). Motivo y cifras en
`reports/06_optimizacion.md`.

In [9]:
stage7 = config.STAGE7_MODEL or rule_winner
assert stage7 in finalists, "config.STAGE7_MODEL debe ser uno de los finalistas"
group_note = config.STAGE7_GROUP_NOTE if stage7 != rule_winner else None
print("Pasa a la etapa 7:", stage7, "· regla:", rule_winner, "·", "EXCEPCIÓN del grupo" if group_note else "según la regla")

Pasa a la etapa 7: lightgbm · regla: random_forest · EXCEPCIÓN del grupo


## 5. Guardar la decisión

In [10]:
from tarifalatam.config import BOOTSTRAP_N, RANDOM_STATE
from tarifalatam.models.tuning import construction_info, save_decision

result = checks[stage7]
decision = {
    "finalista": stage7,
    "parametros": chosen[stage7]["params"],
    "origen_parametros": chosen[stage7]["origin"],
    "comprobacion": {
        "anfitriones_reservados": result["hosts"],
        "anuncios_reservados": result["listings"],
        "mae_partida": result["mae_b"],
        "mae_optimizado": result["mae_a"],
        "diferencia_media": result["mean_difference"],
        "intervalo_95": [result["low"], result["high"]],
        "remuestreos": BOOTSTRAP_N,
        "semilla": RANDOM_STATE,
    },
    "regla": "Se usan los optimizados solo si todo el intervalo está por debajo de cero",
    "comparacion_finalistas": comparison,
    "regla_elegiria": rule_winner,
    "decision_del_grupo": group_note,
    **construction_info(stage7),  # la etapa 7 arma el modelo con make_model (subsample_freq=1, semilla 42)
    "tracking": where,
    "experimento": experiment,
    "ensayos": len(trials),
    "ensayos_por_finalista": N_TRIALS,
}
save_decision(REPORTS_DIR / "06_best_params.json", decision)
print((REPORTS_DIR / "06_best_params.json").read_text(encoding="utf-8"))

{
  "finalista": "lightgbm",
  "parametros": {
    "n_estimators": 522,
    "learning_rate": 0.016724373591399267,
    "num_leaves": 215,
    "min_child_samples": 19,
    "subsample": 0.9281081091680619,
    "colsample_bytree": 0.542468127069249,
    "reg_lambda": 2.8855004473017765
  },
  "origen_parametros": "optimizados",
  "comprobacion": {
    "anfitriones_reservados": 13347,
    "anuncios_reservados": 29023,
    "mae_partida": 38.862347213065604,
    "mae_optimizado": 38.52331318457415,
    "diferencia_media": -0.3390340284914485,
    "intervalo_95": [
      -0.6376654436260816,
      -0.009914320219349446
    ],
    "remuestreos": 1000,
    "semilla": 42
  },
  "regla": "Se usan los optimizados solo si todo el intervalo está por debajo de cero",
  "comparacion_finalistas": {
    "finalista_a": "random_forest",
    "finalista_b": "lightgbm",
    "mae_a": 38.69607178082489,
    "mae_b": 38.52331318457415,
    "diferencia_media": 0.17275859625073786,
    "intervalo_95": [
      -0.

## 6. Comprobar el seguimiento

En una terminal, desde la raíz del repositorio:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow_local.db
```

En <http://127.0.0.1:5000>, el experimento `tarifalatam-optimizacion` debe tener una ejecución por ensayo (por ejemplo
`lightgbm-ensayo-07`), cada una con sus parámetros, sus métricas (`mae`, `mae_std`, `mape`, `rmse`, `bias`, `r2_log`,
`fit_seconds`) y las etiquetas `etapa`, `modelo`, `variables`, `datos` y `estado`. `mlflow_local.db` está en `.gitignore`.